# Building a Multi-Agent RAG Network: From Vector Tuning to AG-UI Integration

Welcome to this step-by-step notebook on constructing a sophisticated custom RAG (Retrieval-Augmented Generation) application. We will emulate the architectural patterns found in enterprise setups, centering around an Orchestrator Agent that delegates tasks via Agent Routing Definitions (ARD) and communicates via the A2A protocol. Finally, we'll expose our agents to the frontend using the AG-UI (Agent-User Interaction) protocol.

## 1. Architectural Overview
*   **AG-UI:** The transport layer bridging our backend agents and the user-facing UI, handling state sync and event streaming.
*   **A2A (Agent-to-Agent):** Standardized protocol for our orchestrator to pass context and delegate tasks to sub-agents.
*   **ARD (Agent Routing Definition):** A manifest system allowing the orchestrator to discover and route requests to specialized agents.
*   **Vector Engines:** A hybrid approach comparing PostgreSQL (`pgvector`) with tunable indexes against purpose-built engines like Qdrant and Milvus.
*   **Orchestration:** Using LangGraph and standard `@tool` decorators for off-the-shelf framework features.

## 2. Setting Up the Vector Databases
When dealing with RAG, the retrieval strategy and underlying index dictate performance and accuracy.

### 2.1 PostgreSQL with `pgvector`: IVF vs. HNSW
PostgreSQL's `pgvector` extension allows us to use relational data alongside vector embeddings. 

*   **IVFFlat:** Groups vectors into clusters (lists). Faster to build, uses less memory.
*   **HNSW:** Builds a multi-layered graph. Higher query performance and recall.

In [ ]:
!pip install psycopg2-binary
import psycopg2

def setup_postgres():
    try:
        # Connect to your Postgres database
        # Replace with your actual connection string
        conn = psycopg2.connect("dbname=rag_db user=admin password=secret host=localhost")
        cur = conn.cursor()
        
        cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")
        
        cur.execute("""
            CREATE TABLE IF NOT EXISTS documents (
                id bigserial PRIMARY KEY,
                content text,
                embedding vector(1536) -- Standard OpenAI embedding dimension
            );
        """)
        
        # Option A: IVF (Inverted File)
        cur.execute("""
            CREATE INDEX IF NOT EXISTS doc_ivf_idx ON documents 
            USING ivfflat (embedding vector_cosine_ops) WITH (lists = 100);
        """)
        
        # Option B: HNSW (Hierarchical Navigable Small World)
        cur.execute("""
            CREATE INDEX IF NOT EXISTS doc_hnsw_idx ON documents 
            USING hnsw (embedding vector_cosine_ops) WITH (m = 16, ef_construction = 64);
        """)
        
        conn.commit()
        print("Postgres vector setup complete.")
    except Exception as e:
        print(f"Postgres not connected. Ensure DB is running to execute: {e}")

setup_postgres()

### 2.2 Optional Behaivor: Qdrant & Milvus
Native vector databases provide advanced filtering, distributed scaling, and built-in hybrid search.

In [ ]:
!pip install qdrant-client pymilvus

from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams

def setup_qdrant():
    try:
        # Qdrant in-memory client for quick local testing
        qdrant = QdrantClient(":memory:")
        qdrant.recreate_collection(
            collection_name="enterprise_docs",
            vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
        )
        print("Qdrant in-memory collection setup complete.")
    except Exception as e:
        print(f"Qdrant Error: {e}")

setup_qdrant()

"""
# MILVUS EXAMPLE (Uncomment and run if Milvus server is available locally/remotely)
from pymilvus import Collection, CollectionSchema, FieldSchema, DataType

fields = [
    FieldSchema(name="id", dtype=DataType.INT64, is_primary=True, auto_id=True),
    FieldSchema(name="vector", dtype=DataType.FLOAT_VECTOR, dim=1536),
    FieldSchema(name="department", dtype=DataType.VARCHAR, max_length=50)
]
schema = CollectionSchema(fields, description="Corporate Documents")
milvus_collection = Collection("corp_docs", schema)

index_params = {
    "metric_type": "COSINE",
    "index_type": "HNSW",
    "params": {"M": 16, "efConstruction": 64}
}
milvus_collection.create_index(field_name="vector", index_params=index_params)
"""
print("Milvus implementation available in comments.")

## 3. Tool Decoration & Agent Definition
We leverage standard LangChain `@tool` decorators for off-the-shelf integration.

In [ ]:
!pip install langchain-core langgraph

from langchain_core.tools import tool
from typing import List

@tool
def retrieve_postgres_data(query_vector: List[float], top_k: int = 5) -> str:
    """
    Searches the standard pgvector database for general knowledge documents.
    Use this for historical records and general corporate policies.
    """
    # Implementation using psycopg2 to query the HNSW index
    return "Retrieved data from Postgres based on HNSW index..."

@tool
def retrieve_qdrant_data(query_vector: List[float], department: str) -> str:
    """
    Performs hybrid search (vector + metadata filtering) on Qdrant.
    Use this when a specific department's context is required.
    """
    # Implementation using Qdrant client
    return "Retrieved data from Qdrant with department filter..."

print("Tools successfully decorated.")

## 4. Orchestrator Agent & Agent Routing Definition (ARD)
The Orchestrator evaluates intent and routes requests to sub-agents via the A2A protocol using LangGraph.

In [ ]:
# 4.1 Defining the ARD
agent_routing_definition = {
    "agents": [
        {
            "id": "agent_general_knowledge",
            "description": "Handles broad queries regarding corporate policies using Postgres pgvector.",
            "tools": ["retrieve_postgres_data"],
            "protocol": "A2A"
        },
        {
            "id": "agent_department_specialist",
            "description": "Handles specialized, department-specific queries using Qdrant hybrid search.",
            "tools": ["retrieve_qdrant_data"],
            "protocol": "A2A"
        }
    ]
}

# 4.2 Building the Orchestrator with LangGraph
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator

# Define our state
class AgentState(TypedDict):
    messages: Annotated[list, operator.add]
    selected_agent: str

def orchestrator_node(state: AgentState):
    last_message = state['messages'][-1].get('content', '')
    # Logic to map intent to ARD (e.g., using an LLM call)
    selected = "agent_general_knowledge" if "policy" in last_message.lower() else "agent_department_specialist"
    return {"selected_agent": selected}

def specialized_agent_node(state: AgentState):
    # The sub-agent runs its own internal tool execution
    response = f"Response from {state['selected_agent']} via A2A."
    return {"messages": [{"role": "assistant", "content": response}]}

# Build the Graph
workflow = StateGraph(AgentState)
workflow.add_node("orchestrator", orchestrator_node)
workflow.add_node("agent_execution", specialized_agent_node)

workflow.set_entry_point("orchestrator")
workflow.add_edge("orchestrator", "agent_execution")
workflow.add_edge("agent_execution", END)

app = workflow.compile()

# Test the workflow locally
test_output = app.invoke({"messages": [{"role": "user", "content": "What is the company policy on remote work?"}]})
print("Workflow output for 'policy' query:", test_output['messages'][-1])

## 5. Exposing the Network via AG-UI
Connecting our LangGraph backend to a frontend via an AG-UI compliant WebSocket.

In [ ]:
!pip install fastapi uvicorn websockets

from fastapi import FastAPI, WebSocket

fast_app = FastAPI()

@fast_app.websocket("/ag-ui-stream")
async def ag_ui_endpoint(websocket: WebSocket):
    await websocket.accept()
    user_input = await websocket.receive_text()
    
    # Emit run started
    await websocket.send_json({"type": "RUN_STARTED"})
    
    # Run the LangGraph Orchestrator and stream output
    for output in app.stream({"messages": [{"role": "user", "content": user_input}]}):
        if "agent_execution" in output:
             content = output["agent_execution"]["messages"][0]["content"]
             await websocket.send_json({
                 "type": "TEXT_MESSAGE_CONTENT",
                 "delta": content
             })
             
    # Emit run finished
    await websocket.send_json({"type": "RUN_FINISHED"})
    await websocket.close()

print("AG-UI FastAPI endpoint defined. Run this app locally using: uvicorn <filename>:fast_app --reload")